## Init

In [ ]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

import sys
import os
from pathlib import Path

# Add project root to path
project_root = str(Path(os.path.abspath('')).parent)
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Set environment variables for DBPATH
os.environ['DB_PATH'] = '/lunarc/nobackup/projects/snic2020-6-41/carl/dev.db'

# Now you can import using the full module path
from scripts.database.db_main import EasyNerDBHandler


db = EasyNerDBHandler()

In [ ]:
db.cache_manager.clear_all()

## Create sankey data

In [ ]:
# get cooccurrences
%autoreload 2
from scripts.database.data_model.entity_cooccurrence import EntityCooccurrence
from scripts.database.data_model.schema import *

In [ ]:
VIEW_DIS_PNM_CO_AGGR_ROW_FACTORY.columns
VIEW_DIS_PNM_CO_AGGR_ROW_FACTORY.refresh(db.cursor)


In [ ]:
db.log_file

In [ ]:
cooccurrences = db.co.get_cooccurrences(e1_norm_id=1, e1_txt_norm_like=None, e2_txt_norm_like=None, min_npmi=0.1, max_npmi=None, min_pmi=0.0, max_pmi=None, min_fq_doc_level=None, max_fq_doc_level=None, limit=100, offset=0)

In [ ]:
len(cooccurrences )

In [ ]:
for co in cooccurrences:
    print(co)

##

In [ ]:
# Generate sankey data from cooccurrence objects
from scripts.database.data_model.entity_cooccurrence import EntityCooccurrence
from scripts.database.data_model.schema import *
VIEW_DIS_PNM_CO_AGGR_ROW_FACTORY.columns

VIEW_DIS_PNM_CO_AGGR_ROW_FACTORY.refresh(db.cursor)

cooccurrences = db.co.get_cooccurrences(min_npmi=0.2, max_npmi=1, min_fq_doc_level=10, limit=100, offset=0)

nodes = list(set())
diseases = list(set())
phenomena = list(set())

links = []
for co in cooccurrences:
    diseases.append(co.e1.txt)
    phenomena.append(co.e2.txt)  # changed from nodes to phenomena
    links.append({
        'source': co.e1.txt,
        'target': co.e2.txt,
        'value': co.npmi
    })

# Sort the lists for deterministic output between runs
diseases.sort()
phenomena.sort()

print (diseases)
print (phenomena)
print (links)


In [ ]:
db.cursor.execute("SELECT COUNT(*) FROM v_NE_VALIDATION_NORMALIZATION WHERE ERROR_ID IS NULL AND OVERLAP = 0 ORDER BY TXT_NORM").fetchone()

In [ ]:
links

In [ ]:
import plotly.graph_objects as go
import numpy as np

# Extract unique diseases and phenomena while preserving order
unique_diseases = []
for disease in diseases:
    if disease not in unique_diseases:
        unique_diseases.append(disease)

unique_phenomena = []
for phenomenon in phenomena:
    if phenomenon not in unique_phenomena:
        unique_phenomena.append(phenomenon)

# Create node labels with categories
node_labels = [f"{d} (Disease)" for d in unique_diseases] + [f"{p} (Phenomenon)" for p in unique_phenomena]

# Create a mapping from entity names to node indices
disease_indices = {d: i for i, d in enumerate(unique_diseases)}
phenomenon_indices = {p: i + len(unique_diseases) for i, p in enumerate(unique_phenomena)}

# Transform links to use indices
link_sources = []
link_targets = []
link_values = []

for link in links:
    source_name = link['source']
    target_name = link['target']

    # Map source to its disease index
    source_idx = disease_indices[source_name]

    # Map target to its phenomenon index
    target_idx = phenomenon_indices[target_name]

    link_sources.append(source_idx)
    link_targets.append(target_idx)
    link_values.append(link['value'])

# Create the Sankey diagram
fig = go.Figure(data=[go.Sankey(
    node=dict(
        pad=15,
        thickness=20,
        line=dict(color="black", width=0.5),
        label=node_labels,
        color=["rgba(31, 119, 180, 0.8)"]*len(unique_diseases) + ["rgba(44, 160, 44, 0.8)"]*len(unique_phenomena)
    ),
    link=dict(
        source=link_sources,
        target=link_targets,
        value=link_values,
        color=["rgba(0, 0, 255, 0.2)"]*len(links),
        hovertemplate='%{source.label} → %{target.label}<br>NPMI: %{value:.4f}<extra></extra>',
    )
)])

# Update layout
fig.update_layout(
    title_text="Disease-Phenomenon Co-occurrence Network",
    font=dict(size=12),
    autosize=True,
    height=1300,
    margin=dict(t=60, l=20, r=20, b=20)
)

# Show the figure
fig.show()

In [ ]:
# Get npmi distribution in db
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import pandas as pd

cursor = db.conn.cursor()
cursor.execute("SELECT npmi FROM v_DIS_PNM_AGGR_ROW_FACTORY WHERE fq_doc_level > 3")
npmis = cursor.fetchall()


In [ ]:
# Convert to numpy array
npmis = np.array(npmis).flatten()
# Plot histogram
plt.figure(figsize=(10, 6))
sns.histplot(npmis, bins=50, kde=True)
plt.title('NPMI Distribution')
plt.xlabel('NPMI')
plt.ylabel('Frequency')
plt.grid()
plt.show()



In [ ]:
# plot distribution of uniq docs
cursor.execute("SELECT uniq_docs FROM v_DIS_PNM_AGGR_ROW_FACTORY WHERE fq_doc_level > 10")
# Convert to numpy array
uniq_docs = np.array(cursor.fetchall()).flatten()
# Plot histogram
plt.figure(figsize=(10, 6))
sns.histplot(uniq_docs, bins=50, kde=True)
plt.title('Unique Documents Distribution')
plt.xlabel('Unique Documents')

In [ ]:
# plot of NPMI and unique documents distributions in one plot with 2 x axes
cursor.execute("SELECT npmi, uniq_docs FROM v_DIS_PNM_AGGR_ROW_FACTORY WHERE fq_doc_level > 5")
data = cursor.fetchall()
# Convert to numpy array
data = np.array(data)
npmis = data[:, 0]
uniq_docs = data[:, 1]

# Create figure and axis
fig_npmi_doc_fq_dist_filtered, ax1 = plt.subplots(figsize=(10, 6))

# Plot NPMI histogram
color = 'tab:blue'
ax1.set_xlabel('NPMI')
ax1.set_ylabel('Frequency', color=color)
sns.histplot(npmis, bins=50, kde=True, ax=ax1, color=color)
ax1.tick_params(axis='y', labelcolor=color)

# Create a second x-axis sharing the same y-axis
ax2 = ax1.twiny()
color = 'tab:green'
ax2.set_xlabel('Unique Documents')
sns.histplot(uniq_docs, bins=50, kde=True, ax=ax2, color=color)
ax2.tick_params(axis='x', labelcolor=color)

# Add grid and title
ax1.grid()
plt.title('NPMI and Unique Documents Distribution')
plt.show()

In [ ]:
# plot of NPMI and unique documents distributions in one plot with 2 x axes
cursor.execute("SELECT npmi, uniq_docs FROM v_DIS_PNM_AGGR_ROW_FACTORY WHERE fq_doc_level > 20")
data = cursor.fetchall()
# Convert to numpy array
data = np.array(data)
npmis = data[:, 0]
uniq_docs = data[:, 1]

# Create figure and axis
fig, ax1 = plt.subplots(figsize=(10, 6))

# Plot NPMI histogram
color = 'tab:blue'
ax1.set_xlabel('NPMI')
ax1.set_ylabel('Frequency', color=color)
sns.histplot(npmis, bins=50, kde=True, ax=ax1, color=color)
ax1.tick_params(axis='y', labelcolor=color)

# Create a second x-axis sharing the same y-axis
ax2 = ax1.twiny()
color = 'tab:green'
ax2.set_xlabel('Unique Documents')
sns.histplot(uniq_docs, bins=50, kde=True, ax=ax2, color=color)
ax2.tick_params(axis='x', labelcolor=color)

# Add grid and title
ax1.grid()
plt.title('NPMI and Unique Documents Distribution')
plt.show()

In [ ]:
# Get top 70 percentile of npmi
cursor.execute("SELECT npmi FROM v_DIS_PNM_AGGR_ROW_FACTORY WHERE fq_doc_level > 2")
npmis = cursor.fetchall()
# Convert to numpy array
npmis = np.array(npmis).flatten()
# Get top 70 percentile
top_70_percentile = np.percentile(npmis, 70)
top_70_percentile
# Get top 70 percentile of uniq docs

In [ ]:
# Create a scatterplot jointplot of NPMI and FQ_DOC_LEVEL
import matplotlib.pyplot as plt
import seaborn as sns

npmis

In [ ]:
%autoreload 2

from scripts.database.statistics.cooccurence_sankey_visualizer import CooccurrenceSankeyVisualizer

# Get cooccurrences from database
cooccurrences = db.co.get_cooccurrences(e1_txt_norm_like="malaria", min_npmi=0.11893167687813, max_npmi=1, min_fq_doc_level=2, limit=300, offset=0)

# Create visualizer instance
sankey_viz = CooccurrenceSankeyVisualizer()

# Create and display the diagram directly from cooccurrences
sankey_viz.create_diagram_from_cooccurrences(
    cooccurrences=cooccurrences,
    title="Disease-Phenomenon Co-occurrence Network"
).display()


In [ ]:
%autoreload 2

from scripts.database.statistics.cooccurence_sankey_visualizer import CooccurrenceSankeyVisualizer

# Get cooccurrences from database
cooccurrences = db.co.get_cooccurrences(e2_txt_norm_like="thunder%", limit=10, offset=0)

# Create visualizer instance
sankey_viz = CooccurrenceSankeyVisualizer()

# Create and display the diagram directly from cooccurrences
sankey_viz.create_diagram_from_cooccurrences(
    cooccurrences=cooccurrences,
    title="Disease-Phenomenon Co-occurrence Network"
).display()

# Alternatively, with method chaining for customization
# sankey_viz.create_diagram(diseases, phenomena, links).customize_layout(height=800, template="plotly_dark").save("disease_phenomenon_cooccurrence.png").display()

In [ ]:
%autoreload 2

from scripts.database.statistics.cooccurence_sankey_visualizer import CooccurrenceSankeyVisualizer

# Get cooccurrences from database
cooccurrences = db.co.get_cooccurrences(e1_txt_norm_like="dengue", max_npmi=1, min_fq_doc_level=40, limit=20, offset=0)

# Create visualizer instance
sankey_viz = CooccurrenceSankeyVisualizer()

# Create and display the diagram directly from cooccurrences
sankey_viz.create_diagram_from_cooccurrences(
    cooccurrences=cooccurrences,
    title="Disease-Phenomenon Co-occurrence Network"
).display()

# Alternatively, with method chaining for customization
# sankey_viz.create_diagram(diseases, phenomena, links).customize_layout(height=800, template="plotly_dark").save("disease_phenomenon_cooccurrence.png").display()